In [ ]:
# ==============================================================================
# UNIVERSIDAD NACIONAL ABIERTA Y A DISTANCIA (UNAD)
# Curso: Programación - Código: 213023
# Grupo: 213023_85
# Fase 4 - Componente práctico - Prácticas simuladas
# Ejercicio 1: Sistema Integral de Gestión de Clientes, Servicios y Reservas
# Empresa: Software FJ
# ==============================================================================
# MÓDULO 1 (Aporte Individual - Estudiante 1):
# Desarrollado por: RAFAEL ANTONIO LLERENA PEREZ
# Contenido del módulo:
# 1. Listas internas en memoria y manejo de archivo de logs (.txt)
# 2. Excepciones personalizadas del sistema
# 3. Clase abstracta general (EntidadSistema)
# 4. Clase Cliente con encapsulación y validaciones estrictas
# 5. Simulaciones 1 y 2 (Registro válido e inválido con manejo de excepciones)
# ==============================================================================

from abc import ABC, abstractmethod
from datetime import datetime

ARCHIVO_LOGS = "logs_software_fj.txt"

lista_clientes = []
lista_servicios = []
lista_reservas = []

# ==============================================================================
# 1. FUNCIÓN PARA EL MANEJO DEL ARCHIVO DE LOGS
# ==============================================================================
def registrar_log(tipo_evento, descripcion):
    try:
        fecha_hora = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        linea = f"[{fecha_hora}] [{tipo_evento}] - {descripcion}\n"
        with open(ARCHIVO_LOGS, "a", encoding="utf-8") as archivo:
            archivo.write(linea)
    except IOError as error_io:
        print(f"Advertencia: No se pudo escribir en el archivo de logs ({error_io})")

# ==============================================================================
# 2. DEFINICIÓN DE EXCEPCIONES PERSONALIZADAS
# ==============================================================================
class SoftwareFJError(Exception):
    def __init__(self, mensaje):
        super().__init__(mensaje)

class ClienteInvalidoError(SoftwareFJError):
    pass

class ParametroFaltanteError(SoftwareFJError):
    pass

class ServicioNoDisponibleError(SoftwareFJError):
    pass

class ReservaInvalidaError(SoftwareFJError):
    pass

class CalculoInconsistenteError(SoftwareFJError):
    pass

# ==============================================================================
# 3. CLASE ABSTRACTA GENERAL (EntidadSistema)
# ==============================================================================
class EntidadSistema(ABC):
    def __init__(self, id_entidad):
        self._id_entidad = id_entidad
        self._fecha_creacion = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    def get_id_entidad(self):
        return self._id_entidad

    @abstractmethod
    def validar_datos(self):
        pass

    @abstractmethod
    def mostrar_informacion(self):
        pass

# ==============================================================================
# 4. CLASE CLIENTE
# ==============================================================================
class Cliente(EntidadSistema):
    def __init__(self, id_cliente, nombre, correo, telefono):
        super().__init__(id_cliente)
        self.__nombre = nombre
        self.__correo = correo
        self.__telefono = telefono
        self.validar_datos()

    def get_nombre(self):
        return self.__nombre

    def set_nombre(self, nuevo_nombre):
        if nuevo_nombre is None or str(nuevo_nombre).strip() == "":
            raise ParametroFaltanteError("El nombre del cliente es un parametro obligatorio y no puede estar vacio.")
        self.__nombre = str(nuevo_nombre).strip()

    def get_correo(self):
        return self.__correo

    def set_correo(self, nuevo_correo):
        if nuevo_correo is None or str(nuevo_correo).strip() == "":
            raise ParametroFaltanteError("El correo electronico del cliente es obligatorio.")
        correo_texto = str(nuevo_correo).strip()
        if "@" not in correo_texto or "." not in correo_texto:
            raise ClienteInvalidoError(f"El correo '{correo_texto}' no tiene un formato valido (debe contener '@' y '.').")
        self.__correo = correo_texto

    def get_telefono(self):
        return self.__telefono

    def set_telefono(self, nuevo_telefono):
        if nuevo_telefono is None or str(nuevo_telefono).strip() == "":
            raise ParametroFaltanteError("El numero de telefono del cliente es obligatorio.")
        telefono_texto = str(nuevo_telefono).strip()
        if not telefono_texto.isdigit() or len(telefono_texto) < 7:
            raise ClienteInvalidoError(f"El telefono '{telefono_texto}' es invalido (debe tener solo numeros y minimo 7 digitos).")
        self.__telefono = telefono_texto

    def validar_datos(self):
        if self._id_entidad is None or str(self._id_entidad).strip() == "":
            raise ParametroFaltanteError("El documento o ID del cliente es obligatorio.")
        self.set_nombre(self.__nombre)
        self.set_correo(self.__correo)
        self.set_telefono(self.__telefono)

    def mostrar_informacion(self):
        return f"Cliente [ID: {self._id_entidad} | Nombre: {self.__nombre} | Correo: {self.__correo} | Teléfono: {self.__telefono}]"

def registrar_cliente(id_cliente, nombre, correo, telefono):
    print(f"\n>>> Procesando solicitud de registro para el cliente: {nombre} (ID: {id_cliente})")
    try:
        for cliente_actual in lista_clientes:
            if str(cliente_actual.get_id_entidad()) == str(id_cliente).strip():
                raise ValueError(f"El ID '{id_cliente}' ya se encuentra asignado a otro cliente en la lista.")
        nuevo_cliente = Cliente(id_cliente, nombre, correo, telefono)
    except (ParametroFaltanteError, ClienteInvalidoError) as error_validacion:
        print(f"[ERROR CONTROLADO]: {error_validacion}")
        registrar_log("ERROR_VALIDACION", f"Fallo en registro de cliente ID '{id_cliente}': {error_validacion}")
        return None
    except ValueError as error_origen:
        try:
            raise ClienteInvalidoError("Operacion rechazada por duplicidad de cliente en el sistema.") from error_origen
        except ClienteInvalidoError as error_encadenado:
            print(f"[ERROR ENCADENADO]: {error_encadenado}")
            print(f" -> Causa original del error: {error_encadenado.__cause__}")
            registrar_log("ERROR_ENCADENADO", f"{error_encadenado} | Causa: {error_encadenado.__cause__}")
            return None
    else:
        lista_clientes.append(nuevo_cliente)
        print(f"[REGISTRO EXITOSO]: {nuevo_cliente.mostrar_informacion()}")
        registrar_log("EVENTO_CLIENTE", f"Registro exitoso: {nuevo_cliente.mostrar_informacion()}")
        return nuevo_cliente
    finally:
        print(f"[FIN DE OPERACIÓN]: Clientes válidos almacenados en memoria: {len(lista_clientes)}")

print("======================================================================")
print(" INICIO DE SIMULACIONES - SISTEMA SOFTWARE FJ (GRUPO 213023_85)")
print("======================================================================")
registrar_log("INICIO_SISTEMA", "Se inicia la ejecucion del Sistema de Gestion Software FJ.")

print("\n--- SIMULACIÓN 1: Registro de Cliente Válido ---")
cliente_1 = registrar_cliente("1001", "Rafael Antonio Llerena Perez", "rllerenap@unadvirtual.edu.co", "3004567890")

print("\n--- SIMULACIÓN 2A: Registro de Cliente Inválido (Correo sin formato válido) ---")
cliente_invalido_1 = registrar_cliente("1002", "Carlos Mendoza", "carlos_mendoza_correo_malo.com", "3109876543")

print("\n--- SIMULACIÓN 2B: Registro de Cliente Inválido (ID duplicado) ---")
cliente_invalido_2 = registrar_cliente("1001", "Usuario Duplicado", "duplicado@correo.com", "3201112233")

# ==============================================================================
# ESPACIOS RESERVADOS PARA EL DESARROLLO COLABORATIVO DE LOS COMPAÑEROS
# ==============================================================================

# ------------------------------------------------------------------------------
# MÓDULO 2 (ESPACIO PARA ESTUDIANTE 2):
# - Crear la Clase Abstracta 'Servicio' (que herede de EntidadSistema) con
#   métodos abstractos para calcular costos, describir servicios y validar parámetros.
# - Crear la Clase hija 'ReservaSala' (herencia y polimorfismo).
# - Realizar las Simulaciones 3 y 4 (creación correcta e incorrecta de ReservaSala).
# ------------------------------------------------------------------------------
# (Escribir código del Estudiante 2 aquí debajo)


# ------------------------------------------------------------------------------
# MÓDULO 3 (ESPACIO PARA ESTUDIANTE 3):
# - Crear las Clases hijas 'AlquilerEquipo' y 'AsesoriaEspecializada' heredando
#   de la clase abstracta 'Servicio', sobrescribiendo métodos y validando parámetros.
# - Realizar las Simulaciones 5 y 6 (creación válida e inválida de estos servicios).
# ------------------------------------------------------------------------------
# (Escribir código del Estudiante 3 aquí debajo)


# ------------------------------------------------------------------------------
# MÓDULO 4 (ESPACIO PARA ESTUDIANTE 4):
# - Implementar métodos sobrecargados para el cálculo de costos (con impuestos,
#   descuentos o parámetros opcionales).
# - Crear la estructura principal de la Clase 'Reserva' (cliente, servicio, duración, estado).
# - Realizar las Simulaciones 7 y 8 (cálculos consistentes e inconsistentes).
# Desarrollado por: HELLER DAVID OROZCO CHARRIS
# ------------------------------------------------------------------------------
class Reserva(EntidadSistema):
    def __init__(self, id_reserva, cliente, servicio, duracion_horas):
        super().__init__(id_reserva)
        self.__cliente = cliente
        self.__servicio = servicio
        self.__duracion_horas = duracion_horas
        self.__estado = "Pendiente"
        self.validar_datos()

    def get_cliente(self):
        return self.__cliente

    def get_servicio(self):
        return self.__servicio

    def get_duracion_horas(self):
        return self.__duracion_horas

    def get_estado(self):
        return self.__estado

    def set_estado(self, nuevo_estado):
        if nuevo_estado not in ["Pendiente", "Confirmada", "Cancelada", "Procesada"]:
            raise ReservaInvalidaError("El estado de la reserva no es válido.")
        self.__estado = nuevo_estado

    def calcular_costo(self, tarifa_base, impuesto_porcentaje=0.0, descuento_monto=0.0):
        if tarifa_base <= 0:
            raise CalculoInconsistenteError(f"Tarifa base inválida ({tarifa_base}): No puede ser menor o igual a cero.")
        if self.__duracion_horas <= 0:
            raise CalculoInconsistenteError(f"Duración inválida ({self.__duracion_horas}h): Debe ser mayor a cero.")
        if descuento_monto < 0:
            raise CalculoInconsistenteError("El descuento no puede ser un valor negativo.")
        subtotal = tarifa_base * self.__duracion_horas
        valor_impuesto = subtotal * (impuesto_porcentaje / 100)
        total_parcial = subtotal + valor_impuesto - descuento_monto
        if total_parcial < 0:
            raise CalculoInconsistenteError(f"Cálculo inconsistente: El descuento aplicado (${descuento_monto}) supera el costo total con impuestos (${subtotal + valor_impuesto}).")
        return total_parcial

    def validar_datos(self):
        if self._id_entidad is None or str(self._id_entidad).strip() == "":
            raise ParametroFaltanteError("El ID de la reserva es obligatorio.")
        if self.__cliente is None:
            raise ParametroFaltanteError("La reserva debe estar asociada a un cliente válido.")
        if self.__duracion_horas <= 0:
            raise CalculoInconsistenteError("La duración de la reserva en horas debe ser mayor a cero.")

    def mostrar_informacion(self):
        return f"Reserva [ID: {self._id_entidad} | Cliente: {self.__cliente.get_nombre()} | Duración: {self.__duracion_horas}h | Estado: {self.__estado}]"

print("\n======================================================================")
print(" INICIO DE SIMULACIONES - MÓDULO 4 (ESTUDIANTE 4: HELLER OROZCO)")
print("======================================================================")
cliente_prueba_m4 = Cliente("2001", "Heller Orozco", "hdorozcoc@unadvirtual.edu.co", "3001234567")

print("\n--- SIMULACIÓN 7: Cálculo de Costo de Reserva Exitoso (Consistente) ---")
try:
    reserva_1 = Reserva("RES-701", cliente_prueba_m4, "Servicio General de Sala", 3)
    tarifa_base_hora = 50000
    iva_porcentaje = 19.0
    descuento_aplicado = 10000
    costo_final = reserva_1.calcular_costo(tarifa_base_hora, iva_porcentaje, descuento_aplicado)
    print(f"[CÁLCULO EXITOSO]: {reserva_1.mostrar_informacion()}")
    print(f" -> Detalles: Tarifa/hora: ${tarifa_base_hora} | Horas: {3} | IVA: {iva_porcentaje}% | Descuento: ${descuento_aplicado}")
    print(f" -> Costo Total Calculado: ${costo_final:.2f}")
    registrar_log("EVENTO_RESERVA", f"Cálculo exitoso para reserva {reserva_1.get_id_entidad()}: Total ${costo_final:.2f}")
except (CalculoInconsistenteError, ParametroFaltanteError, ReservaInvalidaError) as error_mod4:
    print(f"[ERROR EN MÓDULO 4]: {error_mod4}")
    registrar_log("ERROR_MODULO_4", str(error_mod4))

print("\n--- SIMULACIÓN 8: Cálculo de Costo con Error Inconsistente ---")
try:
    reserva_2 = Reserva("RES-702", cliente_prueba_m4, "Asesoría Especializada", 2)
    tarifa_base_hora = 30000
    descuento_exagerado = 200000
    print(">>> Intentando calcular costo con descuento excesivo...")
    costo_erroneo = reserva_2.calcular_costo(tarifa_base_hora, impuesto_porcentaje=19.0, descuento_monto=descuento_exagerado)
except CalculoInconsistenteError as error_inconsistente:
    print(f"[EXCEPCIÓN CONTROLADA]: {error_inconsistente}")
    registrar_log("CALCULO_INCONSISTENTE", f"Fallo en reserva {reserva_2.get_id_entidad()}: {error_inconsistente}")
except Exception as e:
    print(f"[ERROR INESPERADO]: {e}")
finally:
    print("[FIN DE SIMULACIÓN 8]: El sistema procesó la excepción de forma estable.")

print("======================================================================")

# ------------------------------------------------------------------------------
# MÓDULO 5 (ESPACIO PARA ESTUDIANTE 5):
# - Implementar en la Clase 'Reserva' los métodos de confirmación, cancelación y
#   procesamiento con manejo y encadenamiento de excepciones.
# - Realizar las Simulaciones 9 y 10 (reserva exitosa y reserva fallida).
# ------------------------------------------------------------------------------
# (Escribir código del Estudiante 5 aquí debajo)


# ==============================================================================
# VERIFICACIÓN DE LECTURA DEL ARCHIVO DE LOGS GENERADO
# ==============================================================================
print("\n======================================================================")
print(" CONTENIDO ACTUAL DEL ARCHIVO DE LOGS (logs_software_fj.txt)")
print("======================================================================")
try:
    with open(ARCHIVO_LOGS, "r", encoding="utf-8") as archivo_lectura:
        print(archivo_lectura.read())
except FileNotFoundError:
    print("Aun no se ha generado el archivo de logs.")
finally:
    print("======================================================================")
    print(" El sistema finalizó la ejecución de prueba de manera estable.")
    print("======================================================================")